In [ ]:
!pip install requests beautifulsoup4 wikipedia-api

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
import wikipediaapi

In [ ]:
MOOD_TOPICS = {
    "angry":        ["anger management", "stoicism", "forgiveness", "inner peace", "mindfulness"],
    "happy":        ["gratitude", "positive psychology", "joy", "thanksgiving", "celebration"],
    "sad":          ["grief", "healing", "self compassion", "resilience", "hope"],
    "anxious":      ["anxiety relief", "breathing techniques", "calm mind", "worry", "mindfulness"],
    "stressed":     ["stress management", "work life balance", "relaxation", "burnout recovery"],
    "bored":        ["creativity", "curiosity", "new hobbies", "productivity", "learning"],
    "tired":        ["rest and recovery", "sleep hygiene", "energy management", "burnout"],
    "lonely":       ["connection", "community", "friendship", "social wellness", "belonging"],
    "motivated":    ["goal setting", "discipline", "success habits", "growth mindset"],
    "grateful":     ["gratitude practice", "appreciation", "thankfulness", "abundance mindset"],
    "hopeful":      ["optimism", "future thinking", "positive outlook", "dreams and goals"],
    "fearful":      ["overcoming fear", "courage", "confidence building", "facing challenges"],
    "confused":     ["clarity", "decision making", "problem solving", "self reflection"],
    "excited":      ["enthusiasm", "passion", "new beginnings", "adventure"],
    "overwhelmed":  ["simplicity", "minimalism", "prioritization", "focus", "declutter"],
    "jealous":      ["contentment", "self worth", "comparison trap", "personal growth"],
    "guilty":       ["self forgiveness", "making amends", "moving forward", "accountability"],
    "proud":        ["achievement", "self acknowledgment", "milestones", "confidence"],
    "nostalgic":    ["memory", "past reflection", "childhood", "meaningful moments"],
    "inspired":     ["creativity", "innovation", "great thinkers", "philosophy of life"],
    "frustrated":   ["patience", "acceptance", "letting go", "emotional regulation"],
    "calm":         ["meditation", "zen philosophy", "stillness", "present moment"],
    "curious":      ["learning", "science of curiosity", "exploration", "knowledge"],
    "disappointed": ["resilience", "bouncing back", "reframing failure", "perspective"],
    "content":      ["simple living", "satisfaction", "enough", "present moment awareness"]
}

In [ ]:
def scrape_wikipedia(topic, mood, max_articles=5):
    articles = []
    try:
        wiki = wikipediaapi.Wikipedia(
            language='en',
            user_agent='MoodRecommender/1.0'
        )
        page = wiki.page(topic)
        if page.exists():
            articles.append({
                "title": page.title,
                "description": page.summary[:300].strip(),
                "link": page.fullurl,
                "category": topic,
                "mood": mood,
                "source": "wikipedia"
            })
    except Exception as e:
        print(f"Wikipedia error for {topic}: {e}")
    return articles

In [ ]:
def scrape_devto(topic, mood, max_articles=5):
    articles = []
    try:
        url = f"https://dev.to/api/articles?tag={topic.replace(' ', '')}&per_page={max_articles}"
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            for item in data:
                articles.append({
                    "title": item.get("title", ""),
                    "description": item.get("description", "")[:300],
                    "link": item.get("url", ""),
                    "category": topic,
                    "mood": mood,
                    "source": "devto"
                })
    except Exception as e:
        print(f"Dev.to error for {topic}: {e}")
    return articles

In [ ]:
mood_articles = {}

for mood, topics in MOOD_TOPICS.items():
    print(f"Scraping articles for mood: {mood}...")
    mood_articles[mood] = []

    for topic in topics:
        # Wikipedia
        wiki_articles = scrape_wikipedia(topic, mood)
        mood_articles[mood].extend(wiki_articles)
        time.sleep(0.5)  # be polite, don't hammer the server

        # Dev.to
        devto_articles = scrape_devto(topic, mood, max_articles=4)
        mood_articles[mood].extend(devto_articles)
        time.sleep(0.5)

    print(f"  → {len(mood_articles[mood])} articles collected for {mood}")

print("\nDone! Total moods:", len(mood_articles))

Scraping articles for mood: angry...
  → 14 articles collected for angry
Scraping articles for mood: happy...
  → 18 articles collected for happy
Scraping articles for mood: sad...
  → 11 articles collected for sad
Scraping articles for mood: anxious...
  → 7 articles collected for anxious
Scraping articles for mood: stressed...
  → 11 articles collected for stressed
Scraping articles for mood: bored...
  → 20 articles collected for bored
Scraping articles for mood: tired...
  → 8 articles collected for tired
Scraping articles for mood: lonely...
  → 14 articles collected for lonely
Scraping articles for mood: motivated...
  → 11 articles collected for motivated
Scraping articles for mood: grateful...
  → 6 articles collected for grateful
Scraping articles for mood: hopeful...
  → 2 articles collected for hopeful
Scraping articles for mood: fearful...
  → 3 articles collected for fearful
Scraping articles for mood: confused...
  → 18 articles collected for confused
Scraping articles fo

In [ ]:
with open("mood_articles.json", "w", encoding="utf-8") as f:
    json.dump(mood_articles, f, indent=2, ensure_ascii=False)

print("Saved mood_articles.json")
print("Total articles:", sum(len(v) for v in mood_articles.values()))

Saved mood_articles.json
Total articles: 261


In [ ]:
from google.colab import files
files.download("mood_articles.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>